# 08 Train Regression Coverage Model

Train models that predict the actual landed percentage from `0` to `100`, then bucket the prediction into `0`, `1-10`, ..., `91-100`. This is the right shape for granular coverage scoring.

## Setup

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

PROJECT_ROOT = Path.cwd().resolve()
INTERNAL_DATASET_DIR = PROJECT_ROOT / 'data' / 'processed' / 'pr_suggestion_coverage' / 'dataset'
INTERNAL_SCORES_PATH = PROJECT_ROOT / 'reports' / 'metric_scores.csv'
HF_DATASET_DIR = PROJECT_ROOT / 'data' / 'external' / 'github_codereview' / 'dataset'
HF_SCORES_PATH = PROJECT_ROOT / 'data' / 'external' / 'github_codereview' / 'metric_scores.csv'
MODEL_DIR = PROJECT_ROOT / 'models' / 'pr_suggestion_coverage_regression'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
INTERNAL_SAMPLE_WEIGHT = 1.0
HF_SAMPLE_WEIGHT = 0.7
PERCENTAGE_BUCKET_ORDER = ['0', '1-10', '11-20', '21-30', '31-40', '41-50', '51-60', '61-70', '71-80', '81-90', '91-100']

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (10, 5)
MODEL_DIR

## Load Metric Rows

In [ ]:
def read_labels(dataset_dir: Path) -> pd.DataFrame:
    labels = pd.read_csv(dataset_dir / 'labels.csv')
    keep_columns = ['example_id', 'pr_url', 'repo', 'suggestion_source']
    return labels[[column for column in keep_columns if column in labels.columns]].copy()


def load_source_scores(source_name: str, scores_path: Path, dataset_dir: Path, sample_weight: float) -> pd.DataFrame:
    scores = pd.read_csv(scores_path)
    labels = read_labels(dataset_dir)
    rows = scores.merge(labels, on='example_id', how='left', suffixes=('', '_label_file'))
    rows['dataset_source'] = source_name
    rows['sample_weight'] = sample_weight
    rows['repo'] = rows['repo'].fillna(source_name + '/unknown')
    rows['pr_url'] = rows['pr_url'].fillna(rows['repo'].astype(str) + '#unknown-pr')
    rows['group_id'] = rows['dataset_source'] + ':' + rows['pr_url'].astype(str)
    return rows


internal_scores = load_source_scores('internal', INTERNAL_SCORES_PATH, INTERNAL_DATASET_DIR, INTERNAL_SAMPLE_WEIGHT)
hf_scores = load_source_scores('hf_github_codereview', HF_SCORES_PATH, HF_DATASET_DIR, HF_SAMPLE_WEIGHT)
scores = pd.concat([internal_scores, hf_scores], ignore_index=True)

scores['expected_landed_percentage'] = pd.to_numeric(scores['expected_landed_percentage'], errors='coerce')
scores = scores.dropna(subset=['expected_landed_percentage']).copy()
scores['expected_landed_percentage'] = scores['expected_landed_percentage'].clip(0, 100)

print('rows:', len(scores))
print(scores['dataset_source'].value_counts())
print(scores['expected_percentage_bucket'].value_counts().reindex(PERCENTAGE_BUCKET_ORDER).fillna(0).astype(int))
scores.head()

## Feature Schema

In [ ]:
NUMERIC_FEATURES = [
    'line_recall',
    'token_recall',
    'identifier_normalized_token_recall',
    'literal_normalized_token_recall',
    'identifier_and_literal_normalized_token_recall',
    'best_added_line_overlap',
    'best_hunk_token_recall',
    'best_hunk_token_precision',
    'best_hunk_token_f1',
    'best_hunk_identifier_normalized_recall',
    'best_hunk_literal_normalized_recall',
    'best_hunk_identifier_and_literal_normalized_recall',
    'best_hunk_contiguous_line_ratio',
    'best_hunk_token_lcs_recall',
    'best_hunk_size_ratio',
    'meaningful_anchor_recall',
    'meaningful_anchor_count',
    'best_hunk_size',
    'candidate_hunk_count',
    'structural_similarity',
    'structural_node_recall',
    'gumtree_operation_count',
    'gumtree_insert_ratio',
    'gumtree_delete_ratio',
    'gumtree_update_ratio',
    'gumtree_move_ratio',
    'file_overlap_ratio',
    'changed_line_overlap_ratio',
]
BOOLEAN_FEATURES = [
    'exact_normalized_match',
    'structural_available',
    'gumtree_available',
]
CATEGORICAL_FEATURES = [
    'suggestion_language',
    'tokenizer',
    'best_hunk_candidate_type',
    'structural_engine',
    'structural_language',
]

missing_features = [feature for feature in NUMERIC_FEATURES + BOOLEAN_FEATURES + CATEGORICAL_FEATURES if feature not in scores.columns]
if missing_features:
    raise ValueError(f'Missing features: {missing_features}')

for feature in NUMERIC_FEATURES:
    scores[feature] = pd.to_numeric(scores[feature], errors='coerce')
for feature in BOOLEAN_FEATURES:
    scores[feature] = scores[feature].astype(str).str.lower().eq('true').astype(int)
for feature in CATEGORICAL_FEATURES:
    scores[feature] = scores[feature].fillna('none').replace('', 'none').astype(str)

FEATURE_COLUMNS = NUMERIC_FEATURES + BOOLEAN_FEATURES + CATEGORICAL_FEATURES
X = scores[FEATURE_COLUMNS]
y = scores['expected_landed_percentage'].astype(float)
sample_weights = scores['sample_weight'].astype(float)
X.shape, y.describe()

## Source-Aware PR Split

In [ ]:
def source_aware_group_split(rows: pd.DataFrame, test_size: float = 0.25) -> tuple[np.ndarray, np.ndarray]:
    train_parts: list[np.ndarray] = []
    test_parts: list[np.ndarray] = []

    for _, source_rows in rows.groupby('dataset_source', sort=False):
        source_indices = source_rows.index.to_numpy()
        source_groups = source_rows['group_id'].astype(str)

        if source_groups.nunique() < 2 or len(source_rows) < 4:
            fallback_split = max(1, int(round(len(source_rows) * (1 - test_size))))
            train_parts.append(source_indices[:fallback_split])
            test_parts.append(source_indices[fallback_split:])
            continue

        source_splitter = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=RANDOM_STATE)
        train_positions, test_positions = next(
            source_splitter.split(source_rows[FEATURE_COLUMNS], source_rows['expected_landed_percentage'], groups=source_groups)
        )
        train_parts.append(source_indices[train_positions])
        test_parts.append(source_indices[test_positions])

    return np.concatenate(train_parts), np.concatenate(test_parts)


train_index, test_index = source_aware_group_split(scores, test_size=0.25)

X_train = X.loc[train_index]
X_test = X.loc[test_index]
y_train = y.loc[train_index]
y_test = y.loc[test_index]
weights_train = sample_weights.loc[train_index]
test_rows = scores.loc[test_index].copy()

print('train rows:', len(X_train))
print('test rows:', len(X_test))
print('train sources:')
print(scores.loc[train_index, 'dataset_source'].value_counts())
print('test sources:')
print(test_rows['dataset_source'].value_counts())
print('test buckets:')
print(test_rows['expected_percentage_bucket'].value_counts().reindex(PERCENTAGE_BUCKET_ORDER).fillna(0).astype(int))

## Train Regression Candidates

In [ ]:
numeric_preprocessor = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])
categorical_preprocessor = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='none')),
    ('onehot', OneHotEncoder(handle_unknown='ignore')),
])
preprocessor = ColumnTransformer([
    ('numeric', numeric_preprocessor, NUMERIC_FEATURES + BOOLEAN_FEATURES),
    ('categorical', categorical_preprocessor, CATEGORICAL_FEATURES),
])

def make_pipeline(estimator):
    return Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', estimator),
    ])

models = {
    'ridge': make_pipeline(Ridge(alpha=2.0, random_state=RANDOM_STATE)),
    'knn_regressor': make_pipeline(KNeighborsRegressor(n_neighbors=9, weights='distance')),
    'random_forest': make_pipeline(RandomForestRegressor(
        n_estimators=500,
        min_samples_leaf=3,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )),
    'extra_trees': make_pipeline(ExtraTreesRegressor(
        n_estimators=500,
        min_samples_leaf=3,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )),
    'gradient_boosting': make_pipeline(GradientBoostingRegressor(
        n_estimators=250,
        learning_rate=0.04,
        max_depth=3,
        random_state=RANDOM_STATE,
    )),
    'hist_gradient_boosting': make_pipeline(HistGradientBoostingRegressor(
        max_iter=250,
        learning_rate=0.04,
        l2_regularization=0.01,
        random_state=RANDOM_STATE,
    )),
}

trained_models = {}
for name, model in models.items():
    try:
        model.fit(X_train, y_train, regressor__sample_weight=weights_train)
    except TypeError:
        model.fit(X_train, y_train)
    trained_models[name] = model

list(trained_models)

## Evaluate Percentages And Buckets

In [ ]:
def percentage_bucket(value: int | float) -> str:
    percentage = max(0, min(100, int(round(value))))
    if percentage == 0:
        return '0'
    lower_bound = ((percentage - 1) // 10) * 10 + 1
    upper_bound = min(lower_bound + 9, 100)
    return f'{lower_bound}-{upper_bound}'


def bucket_codes(values: pd.Series | np.ndarray | list) -> np.ndarray:
    buckets = [percentage_bucket(value) for value in values]
    return pd.Categorical(buckets, categories=PERCENTAGE_BUCKET_ORDER, ordered=True).codes


def clipped_predictions(predictions: np.ndarray) -> np.ndarray:
    return np.clip(np.asarray(predictions, dtype=float), 0, 100)


def evaluate_percentage_predictions(actual: pd.Series, predicted: np.ndarray) -> dict:
    predicted = clipped_predictions(predicted)
    actual_values = actual.to_numpy(dtype=float)
    actual_bucket_codes = bucket_codes(actual_values)
    predicted_bucket_codes = bucket_codes(predicted)
    dangerous_high = np.logical_and(predicted >= 80, actual_values <= 20)
    dangerous_low = np.logical_and(predicted <= 20, actual_values >= 80)
    return {
        'percentage_mae': float(mean_absolute_error(actual_values, predicted)),
        'percentage_rmse': float(mean_squared_error(actual_values, predicted) ** 0.5),
        'r2': float(r2_score(actual_values, predicted)),
        'bucket_exact_accuracy': float(np.mean(actual_bucket_codes == predicted_bucket_codes)),
        'bucket_within_1_accuracy': float(np.mean(np.abs(actual_bucket_codes - predicted_bucket_codes) <= 1)),
        'dangerous_error_rate': float(np.mean(np.logical_or(dangerous_high, dangerous_low))),
    }


evaluation_rows = []
evaluation_report = {'bucket_order': PERCENTAGE_BUCKET_ORDER, 'models': {}}

rule_predictions = test_rows['predicted_percentage'].to_numpy(dtype=float)
rule_metrics = evaluate_percentage_predictions(y_test, rule_predictions)
evaluation_report['models']['rule_metric_percentage'] = rule_metrics
evaluation_rows.append({'model': 'rule_metric_percentage', **rule_metrics})

for name, model in trained_models.items():
    predictions = model.predict(X_test)
    metrics = evaluate_percentage_predictions(y_test, predictions)
    evaluation_report['models'][name] = metrics
    evaluation_rows.append({'model': name, **metrics})

evaluation = pd.DataFrame(evaluation_rows).sort_values(
    ['percentage_mae', 'dangerous_error_rate', 'bucket_within_1_accuracy'],
    ascending=[True, True, False],
)
evaluation

## Visual Comparison

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
plot_metrics = [
    ('percentage_mae', 'MAE lower is better'),
    ('percentage_rmse', 'RMSE lower is better'),
    ('bucket_exact_accuracy', 'Bucket exact higher is better'),
    ('bucket_within_1_accuracy', 'Bucket ±1 higher is better'),
]
for ax, (metric, title) in zip(axes, plot_metrics):
    sns.barplot(data=evaluation, x='model', y=metric, ax=ax, color='#4C78A8')
    ax.set_title(title)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=45)
    if 'accuracy' in metric:
        ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

## Per-Source Evaluation

In [ ]:
per_source_rows = []
for source_name, source_rows in test_rows.groupby('dataset_source'):
    source_actual = source_rows['expected_landed_percentage'].astype(float)

    rule_metrics = evaluate_percentage_predictions(source_actual, source_rows['predicted_percentage'].to_numpy(dtype=float))
    per_source_rows.append({'dataset_source': source_name, 'predictor': 'rule_metric_percentage', 'rows': len(source_rows), **rule_metrics})

    for model_name, model in trained_models.items():
        source_predictions = model.predict(source_rows[FEATURE_COLUMNS])
        model_metrics = evaluate_percentage_predictions(source_actual, source_predictions)
        per_source_rows.append({'dataset_source': source_name, 'predictor': model_name, 'rows': len(source_rows), **model_metrics})

per_source_evaluation = pd.DataFrame(per_source_rows).sort_values(['dataset_source', 'percentage_mae'])

internal_trained_scores = per_source_evaluation[
    (per_source_evaluation['dataset_source'] == 'internal')
    & (per_source_evaluation['predictor'] != 'rule_metric_percentage')
].copy()
if internal_trained_scores.empty:
    selected_trained_model_name = evaluation[evaluation['model'] != 'rule_metric_percentage'].iloc[0]['model']
else:
    selected_trained_model_name = internal_trained_scores.sort_values(
        ['percentage_mae', 'dangerous_error_rate', 'bucket_within_1_accuracy'],
        ascending=[True, True, False],
    ).iloc[0]['predictor']

best_model_name = selected_trained_model_name
best_model = trained_models[best_model_name]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, metric, title in zip(
    axes,
    ['percentage_mae', 'bucket_exact_accuracy', 'bucket_within_1_accuracy'],
    ['MAE By Source', 'Bucket Exact By Source', 'Bucket ±1 By Source'],
):
    sns.barplot(data=per_source_evaluation, x='dataset_source', y=metric, hue='predictor', ax=ax)
    ax.set_title(title)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=15)
    if 'accuracy' in metric:
        ax.set_ylim(0, 1)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

print('selected trained model by internal validation:', selected_trained_model_name)
per_source_evaluation


## Bucket Confusion For Best Model

In [ ]:
if best_model is None:
    best_predictions = rule_predictions
else:
    best_predictions = clipped_predictions(best_model.predict(X_test))

test_rows['regression_prediction'] = np.rint(best_predictions).astype(int)
test_rows['regression_bucket'] = [percentage_bucket(value) for value in test_rows['regression_prediction']]
test_rows['expected_bucket'] = [percentage_bucket(value) for value in test_rows['expected_landed_percentage']]
test_rows['regression_absolute_error'] = (test_rows['expected_landed_percentage'] - test_rows['regression_prediction']).abs()

bucket_confusion = pd.crosstab(test_rows['expected_bucket'], test_rows['regression_bucket']).reindex(
    index=PERCENTAGE_BUCKET_ORDER,
    columns=PERCENTAGE_BUCKET_ORDER,
    fill_value=0,
)
fig, ax = plt.subplots(figsize=(11, 8))
sns.heatmap(bucket_confusion, annot=True, fmt='d', cmap='Blues', ax=ax)
ax.set_title(f'Bucket Confusion: {best_model_name}')
ax.set_xlabel('Predicted bucket')
ax.set_ylabel('Expected bucket')
plt.tight_layout()
plt.show()

bucket_confusion

## Prediction Scatter

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
sns.scatterplot(
    data=test_rows,
    x='expected_landed_percentage',
    y='regression_prediction',
    hue='dataset_source',
    style='label',
    s=80,
    ax=ax,
)
ax.plot([0, 100], [0, 100], color='black', linestyle='--', linewidth=1)
ax.set_xlim(-3, 103)
ax.set_ylim(-3, 103)
ax.set_title(f'Expected vs Predicted Percentage: {best_model_name}')
ax.set_xlabel('Expected percentage')
ax.set_ylabel('Predicted percentage')
plt.tight_layout()
plt.show()

## Worst Errors

In [ ]:
worst_columns = [
    'example_id',
    'dataset_source',
    'repo',
    'label',
    'expected_landed_percentage',
    'expected_bucket',
    'predicted_percentage',
    'predicted_percentage_bucket',
    'regression_prediction',
    'regression_bucket',
    'regression_absolute_error',
    'suggestion_language',
    'best_hunk_token_f1',
    'meaningful_anchor_recall',
    'changed_line_overlap_ratio',
]
test_rows.sort_values('regression_absolute_error', ascending=False)[worst_columns].head(30)

## Save Best Regression Model

In [ ]:
best_trained_model_name = selected_trained_model_name
best_trained_model = trained_models[best_trained_model_name]

model_path = MODEL_DIR / 'model.joblib'
schema_path = MODEL_DIR / 'feature_schema.json'
report_path = MODEL_DIR / 'evaluation_report.json'

joblib.dump(best_trained_model, model_path)
feature_schema = {
    'model_name': best_trained_model_name,
    'selection_policy': 'lowest internal validation percentage_mae among trained regressors',
    'prediction_type': 'percentage_regression',
    'bucket_order': PERCENTAGE_BUCKET_ORDER,
    'numeric_features': NUMERIC_FEATURES,
    'boolean_features': BOOLEAN_FEATURES,
    'categorical_features': CATEGORICAL_FEATURES,
    'feature_columns': FEATURE_COLUMNS,
    'prediction_input': 'deterministic metric row derived from suggestion + merged diff/code',
    'prediction_output': '0-100 percentage plus reporting bucket',
}
schema_path.write_text(json.dumps(feature_schema, indent=2), encoding='utf-8')

evaluation_report['best_overall_by_mae'] = str(evaluation.iloc[0]['model'])
evaluation_report['saved_trained_model'] = str(best_trained_model_name)
evaluation_report['selection_policy'] = feature_schema['selection_policy']
evaluation_report['evaluation_table'] = evaluation.to_dict(orient='records')
evaluation_report['per_source_evaluation'] = per_source_evaluation.to_dict(orient='records')
report_path.write_text(json.dumps(evaluation_report, indent=2), encoding='utf-8')

model_path, schema_path, report_path


## Note On KNN

`knn_regressor` is included as a candidate. It is useful as a similarity baseline, but it should not be the only production model unless it clearly wins on internal validation. True semi-supervised KNN would require a separate unlabeled pool and a pseudo-labeling policy; this notebook keeps the first version measurable and conservative.